# Inheritance

Focused on your two assigned sections: "The super() Function" and "Multiple Inheritance and Method Resolution Order"

## Core Idea

This chapter is a deliberately skeptical look at inheritance. The two 
sections you're reading build up to one conclusion: multiple inheritance 
creates real ambiguity (which method actually runs when two parent classes 
define the same name?), Python resolves that ambiguity with a well-defined 
but non-obvious algorithm (MRO), and — after seeing how much complexity 
that creates — the chapter's actual recommendation is to prefer 
**composition over inheritance** in most real designs. 

## super() — key points

- `super()` doesn't mean "my direct parent" — it means **"the next class in 
the Method Resolution Order, after this one."** In multiple inheritance, 
that's not always obvious just by looking at the class definition.
- **Why `super()` is recommended over calling a specific class directly** 
(e.g. `A.ping(self)`): `super()` automatically follows the MRO, so it stays 
correct even if the class hierarchy changes later. Hardcoding `A.ping(self)` 
works but becomes fragile — you're now stuck referencing a specific class, 
even if the hierarchy gets restructured.
- You *can* bypass `super()`/MRO entirely and call a method directly on a 
specific class (`C.pong(self)`) — this is legal and sometimes useful, but 
you must pass `self` explicitly, since you're calling an "unbound" method 
directly off the class rather than through an instance.

In [ ]:
class A:
    def ping(self):
        print('ping:', self)

class B(A):
    def pong(self):
        print('pong:', self)

class C(A):
    def pong(self):
        print('PONG:', self)   # uppercase — deliberately different from B's

class D(B, C):
    def ping(self):
        super().ping()             # delegate up the chain, don't hardcode which class
        print('post-ping:', self)

    def pingpong(self):
        self.ping()        # uses D's own ping
        super().ping()      # skips D, goes straight to A's ping
        self.pong()          # follows MRO -> finds B's pong
        super().pong()        # also follows MRO -> also finds B's pong
        C.pong(self)            # explicitly bypasses MRO, forces C's pong

## MRO — the essentials

- **The problem ("diamond problem")**: when a class inherits from two 
classes that both define a method with the same name, which one wins?
- **The answer**: Python computes a fixed search order (the MRO), stored on 
every class as `ClassName.__mro__`. When you call `d.pong()`, Python walks 
this tuple **left to right** and uses the **first** matching implementation 
it finds.
- **What determines the order**: primarily the order superclasses are 
listed in the class definition. `class D(B, C):` searches B before C. If 
you'd written `class D(C, B):` instead, C would be searched first — same 
classes, different declared order, different MRO.
- **You rarely need to compute MRO by hand** — just inspect it directly:
```python
def print_mro(cls):
    print(', '.join(c.__name__ for c in cls.__mro__))

print_mro(bool)
# bool, int, object
```
- The algorithm behind this is called **C3 linearization** — the book's own 
advice: don't worry about the algorithm's internals unless you're doing 
heavy multiple inheritance with non-trivial hierarchies. Knowing 
`ClassName.__mro__` exists and is inspectable is enough for practical work.

In [ ]:
d = D()
d.pong()     # → "pong: ..." — runs B's version, not C's

D.__mro__
# (D, B, C, A, object)

## Why this matters for your Pipeline exercise

After walking through the diamond problem and MRO, the chapter's actual 
conclusion (quoting the *Design Patterns* book directly) is:

> "Favor Object Composition over Class Inheritance."

**The reasoning, condensed:**
- Inheritance is a form of **tight coupling** — a subclass is deeply 
entangled with its parent's internals, and tall/multiple inheritance trees 
become brittle and hard to reason about (exactly what you just saw with 
the diamond problem).
- **Composition** — an object *holding a reference to* another object and 
delegating to it, rather than inheriting from it — achieves code reuse 
without that tight coupling. You can swap out or extend behavior by 
changing what an object *holds*, without touching any class hierarchy.
- Two questions worth asking before reaching for inheritance: is this an 
**"is-a" relationship** (→ inheritance may be justified) or is this really 
just **code reuse** (→ composition is usually better)? A `Pipeline` is not 
"a kind of" `Transform` — it *uses* transforms. That's composition, not 
inheritance — exactly why the exercise specifies "the `Pipeline` *has* 
transforms, rather than inheriting from them."

# EXERCISES

- Practice:
  1. Design a base class `Transform` with a method `apply(self, data)` that raises `NotImplementedError`.
  2. Write 2 subclasses: `Normalize` and `ClipOutliers`, each implementing `apply()` differently (can work on a plain Python list of numbers, no need for numpy yet).
  3. Write a `Pipeline` class that holds a list of `Transform` objects and applies them in sequence via composition (the `Pipeline` *has* transforms, rather than inheriting from them).

Imagine you're processing a demand profile (hourly electricity consumption over a year) before feeding it into a simulation. Realistically, you don't just use the raw data — you probably need to:

- Clip outliers — a sensor glitch reported 50,000 kW for one hour; cap it at a sane maximum.
- Normalize — scale everything to a 0–1 range for the optimizer.
- Maybe later: fill missing values, smooth spikes, resample to a different time resolution...

In [ ]:
# The naive way — and why it gets bad fast
"""

data = clip_outliers(raw_data, lower=0, upper=1000)
data = normalize(data)
data = fill_missing(data)

"""

This works, at first. But real projects don't stay this simple:

- What if different datasets need different steps, in different orders? (PV data needs clipping but not normalization; demand data needs both.)
- What if you want to log each step, time each step, or turn steps on/off via config?
- What if a colleague needs to reuse just "your cleaning process" as one unit, without knowing which three functions to call in which order?

## The Pipeline idea — package the steps themselves as objects, then hold a list of them

Instead of hardcoding a sequence of function calls, you define each step as an object with a consistent interface (apply()), and a Pipeline that just holds a list of these steps and runs them in order:

In [2]:
class Transform:
    def apply(self, data):
        raise NotImplementedError

# Transform exists to enforce a shared interface (apply() must exist), not to store data — so it has no __init__ because it has nothing to initialize. 
# Subclasses get an __init__ only when they individually need to remember something specific to configure their own behavior later.

# apply() - "take some data in, and give back a transformed version of that data." It's the one required action every Transform subclass must provide, following the contract from Transform.

class Normalize(Transform):
    def apply(self, data):
        min_val, max_val = min(data), max(data)
        return [(x - min_val) / (max_val - min_val) for x in data]

class ClipOutliers(Transform):
    def __init__(self, lower, upper):
        self.lower = lower
        self.upper = upper

    def apply(self, data):
        return [max(self.lower, min(self.upper, x)) for x in data]

class Pipeline:
    def __init__(self, transforms):
        self.transforms = transforms   # composition: Pipeline HAS transforms

    def run(self, data):
        for t in self.transforms:
            data = t.apply(data)
        return data

In [3]:
pipeline = Pipeline([ClipOutliers(lower=0, upper=100), Normalize()])
pipeline.run([10, 9999, -5])

[0.1, 1.0, 0.0]

## Notes on this design

- `Transform.apply()` raising `NotImplementedError` is a common pattern for 
defining an **interface**: it says "any subclass must implement this," 
without `Transform` itself needing `abc.ABC` machinery (which the book 
mentions as the more formal way to do this, covered in Ch. 13).
- `Normalize` and `ClipOutliers` both **inherit** from `Transform` (an 
"is-a" relationship — each one genuinely *is a* Transform, sharing the 
same interface) — but `Pipeline` **composes** a list of them (a "has-a" 
relationship) rather than inheriting from `Transform` itself. This mirrors 
exactly the interface-inheritance-vs-composition distinction from the 
reading.
- This is why the pattern is flexible: adding a new transform type later 
means writing one new subclass — the `Pipeline` class itself never needs 
to change.